# English-Hindi Neural Machine Translation Model Report

In [1]:
!pip install datasets transformers sentencepiece sacrebleu evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.0 MB/s eta 0:00:00


First, we install all the required Python packages for this project, such as `datasets`, `transformers`, `sentencepiece`, `sacrebleu`, `evaluate`, and `accelerate`. These libraries are essential for working with Hugging Face models, data processing, and evaluation.

## 1. Setup and Data Loading

This section handles the initial setup, including installing necessary libraries and loading the English-Hindi translation dataset from Hugging Face.

In [2]:
from datasets import load_dataset
dataset = load_dataset("cfilt/iitb-english-hindi")

README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/85.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/500k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

The `cfilt/iitb-english-hindi` dataset is loaded from the Hugging Face `datasets` library. This dataset provides parallel English and Hindi sentences, which are crucial for training a translation model.

In [3]:
max_length = 256

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-hi")
model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-hi")

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/306M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

We initialize the tokenizer and a sequence-to-sequence model (`AutoModelForSeq2SeqLM`) from `Helsinki-NLP/opus-mt-en-hi`. This model is a MarianMT model fine-tuned for English-to-Hindi translation. A `max_length` of 256 is set for tokenization and generation.

## 2. Model and Tokenizer Initialization

Here, we load a pre-trained English-Hindi translation model and its corresponding tokenizer from the Hugging Face Hub.

In [4]:
article = dataset['validation'][2]['translation']['en']
inputs = tokenizer(article, return_tensors="pt")

translated_tokens = model.generate(
    **inputs, max_length=256
)
tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

'एमएनएपी शिक्षकों के राष्ट्रपति, राजस्वीवर ने इस पुरस्कार को पेश करने के द्वारा स्कूल की प्रतिष्ठा की.'

An example English sentence from the validation set is tokenized and passed to the model for translation. The `model.generate` method produces the Hindi translation, which is then decoded back into a human-readable string. This shows the initial performance of the pre-trained model.

## 3. Initial Translation Demonstration

Before fine-tuning, we demonstrate the model's out-of-the-box translation capability.

In [5]:
dataset['validation'][2]['translation']['hi']

'मनपा शिक्षक संघ के अध्यक्ष राजेश गवरे ने स्कूल को भेंट देकर सराहना की।'

This cell displays the ground truth Hindi translation for comparison with the model's output.

In [6]:
def preprocess_function(examples):
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["hi"] for ex in examples["translation"]]

    model_inputs = tokenizer(inputs, max_length=max_length, truncation=True)
    labels = tokenizer(targets, max_length=max_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

The `preprocess_function` takes English sentences as inputs and Hindi sentences as targets. It tokenizes both, ensuring they are truncated to `max_length` (256) if longer. The tokenized target sentences are assigned to the `labels` key, which is required for sequence-to-sequence training.

## 4. Data Preprocessing

This section defines and applies a preprocessing function to prepare the dataset for training.

In [7]:
tokenized_datasets_validation = dataset['validation'].map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["validation"].column_names,
    batch_size=2
)

tokenized_datasets_test = dataset['test'].map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["test"].column_names,
    batch_size=2)

Map:   0%|          | 0/520 [00:00<?, ? examples/s]

Map:   0%|          | 0/2507 [00:00<?, ? examples/s]

The `preprocess_function` is applied to the validation and test splits of the dataset using the `map` method. This efficiently processes the entire dataset in batches. The original `translation` columns are removed as they are no longer needed after tokenization, and `batch_size=2` is used for processing.

In [8]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

A `DataCollatorForSeq2Seq` is initialized with the tokenizer and model. This collator is crucial for batching, as it handles padding of sequences to the maximum length within each batch, and also prepares the `decoder_input_ids` from the `labels`.

## 5. Data Collator

The data collator is used to dynamically pad sequences to the longest length in a batch during training.

In [9]:
for parameter in model.parameters():
    parameter.requires_grad = True
num_layers_to_freeze = 10
for layer_index, layer in enumerate(model.model.encoder.layers):
    print
    if layer_index < len(model.model.encoder.layers) - num_layers_to_freeze:
        for parameter in layer.parameters():
            parameter.requires_grad = False

num_layers_to_freeze = 10
for layer_index, layer in enumerate(model.model.decoder.layers):
    print
    if layer_index < len(model.model.encoder.layers) - num_layers_to_freeze:
        for parameter in layer.parameters():
            parameter.requires_grad = False

This section iterates through the encoder and decoder layers of the model. It sets `requires_grad = False` for the initial layers (all but the last `num_layers_to_freeze` layers), effectively freezing their weights during fine-tuning. This allows only the later layers to be updated, which can be beneficial for transfer learning on smaller datasets.

## 6. Model Layer Freezing

To optimize training and potentially prevent catastrophic forgetting, some layers of the pre-trained model are frozen.

In [10]:
import evaluate

metric = evaluate.load("sacrebleu")

import numpy as np


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

The `compute_metrics` function uses the `sacrebleu` metric from the `evaluate` library. It decodes the model's predictions and the true labels (handling special tokens and padding), then computes the BLEU score, which is a common metric for machine translation quality.

## 7. Metric Computation

This function defines how evaluation metrics (specifically BLEU score) will be computed during training.

In [11]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import Seq2SeqTrainingArguments

model.to(device)
training_args = Seq2SeqTrainingArguments(
    f"finetuned-nlp-en-hi",
    gradient_checkpointing=True,
    per_device_train_batch_size=32,
    learning_rate=1e-5,
    warmup_steps=2,
    max_steps=2000,
    fp16=True,
    optim='adafactor',
    per_device_eval_batch_size=16,
    metric_for_best_model="eval_bleu",
    predict_with_generate=True,
    push_to_hub=False,
)

The model is moved to the appropriate device (CUDA if available, otherwise CPU). `Seq2SeqTrainingArguments` are defined to configure various aspects of the training process, such as batch size, learning rate, optimization strategy (`adafactor`), mixed-precision training (`fp16`), and evaluation metrics. `gradient_checkpointing` is enabled to save memory.

## 8. Training Setup

Here, we configure the training arguments and initialize the `Seq2SeqTrainer`.

In [12]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets_test,
    eval_dataset=tokenized_datasets_validation,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Step,Training Loss
500,3.226470
1000,2.539384
1500,2.350377
2000,2.286338


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2000, training_loss=2.6006421508789064, metrics={'train_runtime': 863.6796, 'train_samples_per_second': 74.102, 'train_steps_per_second': 2.316, 'total_flos': 1107133575266304.0, 'train_loss': 2.6006421508789064, 'epoch': 25.31645569620253})

A `Seq2SeqTrainer` is instantiated with the model, training arguments, tokenized datasets (using `test` for training and `validation` for evaluation, which might be a typo in the original code and usually would be `train_dataset=tokenized_datasets_train`), data collator, and the `compute_metrics` function. Finally, `trainer.train()` initiates the fine-tuning process.

In [13]:
import gradio as gr


def translate(text):
  inputs = tokenizer(text, return_tensors="pt").to(device)
  translated_tokens = model.generate(**inputs,  max_length=256)
  results = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
  return results

interface = gr.Interface(fn=translate,inputs=gr.Textbox(lines=2, placeholder='Text to translate'),
                        outputs='text')

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://700a4bb2452c02a3ce.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


A `translate` function is defined which takes input text, tokenizes it, generates a translation using the fine-tuned model, and decodes the output. A Gradio interface is then created, linking the `translate` function to a text input and text output field. `interface.launch()` starts the Gradio web server, making the translation model accessible via a local or public URL.

## 9. Gradio Interface for Live Translation

After training, a simple web interface is created using Gradio to allow interactive translation.